# C의 공통 문단 적재·자연어 트리 준비
C는 같은 corpus의 원문을 Jev `Choice`로 분류한다. 질문·정답 라벨·임베딩은 입력하지 않는다.
사람이 정의한 [분류 설명](../config/c_taxonomy.json)을 초기 공간으로 사용한다. 원문과 원문 위치는 공통 저장소에 보존한다.
여러 가능성이 있는 배치는 Choice 확률 기준으로 복수 위치에 참조한다. 이 확률은 독립적인 다중 라벨 확률이 아니다.
같은 설정의 완성된 트리는 무결성을 확인하고 재사용한다. 준비 API 사용량·시간·비용과 사람의 미측정 작업 시간을 구분한다.

### 1. 환경과 초기 준비 조건

In [ ]:
from pathlib import Path
import sys, os, json, time
from dotenv import load_dotenv
ROOT = Path.cwd()
if not (ROOT/'config/c.json').exists(): ROOT = ROOT.parent
assert (ROOT/'config/c.json').exists()
load_dotenv(ROOT/'.env', override=False, encoding='utf-8-sig')
sys.path.insert(0, str(ROOT))
import pandas as pd
import httpx
from contextlib import nullcontext
from rag_baselines.raw_logging import RawLog
from rag_baselines.data import read_json, file_sha256, write_json
from rag_baselines.store import DocumentStore
from rag_baselines.c_tree import taxonomy_nodes, tree_directory, load_tree, prepare_tree
from rag_baselines.jev import JevClient
C_CONFIG = read_json(ROOT/'config/c.json')
TAXONOMY = read_json(ROOT/'config/c_taxonomy.json')
print('Workspace:', ROOT)
print('Jev key configured:', bool(os.environ.get('TYPESAFE_API_KEY')))


In [ ]:
RUN_PREPARATION = True
CORPUS_SIZE = None                  # None=공통 N0; 10000 또는 40000도 가능
MAX_PREP_API_CALLS = 512             # 이번 준비 실행의 실제 호출 한도; 실패 후 완료한 배치는 재사용
manifest = read_json(ROOT/'data/prepared/manifest.json')
corpus_size = CORPUS_SIZE or manifest['n0']
entry = manifest['corpora'][str(corpus_size)]
corpus_path = ROOT/entry['path']
assert file_sha256(corpus_path) == entry['sha256']
store = DocumentStore(corpus_path)
_, leaf_descriptions = taxonomy_nodes(TAXONOMY)
directory = tree_directory(ROOT, store, TAXONOMY, C_CONFIG)
display(pd.DataFrame([{'id':key, **value} for key,value in leaf_descriptions.items()]))
print('Original passages:', len(store.passages), '| taxonomy leaves:', len(leaf_descriptions))
print('Pinned Jev model:', C_CONFIG['model'])
print('Prepared directory:', directory)

SAVE_RAW_LOGS = read_json(ROOT/'config/experiment.json').get('save_raw_logs',True)
if type(SAVE_RAW_LOGS) is not bool: raise ValueError('save_raw_logs must be true or false.')
RAW_LOG = RawLog(ROOT,manifest={'notebook':'04_prepare_c_tree.ipynb','corpus_sha256':store.sha256,
    'c_config':C_CONFIG,'max_prep_api_calls':MAX_PREP_API_CALLS},prices={'jev':C_CONFIG['prices']}) if SAVE_RAW_LOGS else None
if RAW_LOG:
    for path in (ROOT/'config/c.json',ROOT/'config/c_taxonomy.json',ROOT/'config/experiment.json',corpus_path,
                 ROOT/'data/prepared/manifest.json',ROOT/'notebook/04_prepare_c_tree.ipynb',
                 ROOT/'rag_baselines/c_tree.py',ROOT/'rag_baselines/jev.py',ROOT/'rag_baselines/raw_logging.py'):
        RAW_LOG.snapshot(path)
    print('Preparation raw logs:',RAW_LOG.directory)


### 2. 원문 자동 적재와 재개
완성된 트리가 있으면 API 호출 없이 검증한다. 새 트리는 원문별 Choice 질문을 묶어서 호출한다.
모든 원문을 빠뜨리지 않고 배치하며, 예외가 생겨도 완료된 응답과 알려진 사용량은 checkpoint에 남긴다.
24단어 발췌와 제목을 분기 탐색용 카드로 만들며, 점수화·최종 반환에는 전체 원문을 쓴다.
byte guard로 요청을 분할한다. Jev 토크나이저가 공개되어 있지 않아 정확한 입력 토큰은 응답 usage만 사용한다.
`save_raw_logs`는 공통 설정을 따른다. 요청 전 본문·실패 응답도 `results/raw/`에 남긴다. 같은 트리의 과거 준비 비용을 이번 새 호출 비용에 더하지 않는다.

In [ ]:
tree = None
preparation = None
with RAW_LOG.scope(phase='preparation',method='C') if RAW_LOG else nullcontext():
    try:
        if (directory/'manifest.json').exists():
            tree, preparation = load_tree(ROOT, store, TAXONOMY, C_CONFIG)
        elif RUN_PREPARATION:
            if not os.environ.get('TYPESAFE_API_KEY'):
                raise RuntimeError('루트 .env에 TYPESAFE_API_KEY를 등록하고 커널을 다시 시작하세요.')
            with httpx.Client(timeout=C_CONFIG['timeout_seconds'],event_hooks=RAW_LOG.http_hooks if RAW_LOG else {}) as prep_http:
                with JevClient(C_CONFIG,http_client=prep_http) as client:
                    tree, preparation = prepare_tree(ROOT, corpus_path, TAXONOMY, C_CONFIG,
                                                      client=client, max_calls=MAX_PREP_API_CALLS)
        else:
            print('트리 준비 미실행. 새 트리는 RUN_PREPARATION=True와 Jev 키가 필요합니다.')
        if RAW_LOG and preparation:
            RAW_LOG.result('tree_preparation',preparation)
            RAW_LOG.event('tree','cache_status',{'cache_hit':preparation['cache_hit'],
                'accounting':'manifest usage includes historical preparation; ledger counts this session only'})
    finally:
        if RAW_LOG:
            for name in ('tree.json','manifest.json','checkpoint.json','checkpoint_hash.json'):
                path = directory/name
                if path.exists(): RAW_LOG.snapshot(path)
if preparation:
    display(pd.DataFrame([{key:preparation[key] for key in
        ('cache_hit','prepare_seconds','corpus_count','node_count','placement_count','taxonomy_leaf_count',
         'api_cost_usd','human_taxonomy_authoring_seconds','generated_summary_api_calls')}]))
    display(pd.DataFrame([preparation['api_usage']]))
    print('Known usage complete:', preparation['api_usage']['usage_complete'])

### 3. 전체 corpus 접근 가능성과 원문 검증

In [ ]:
if tree:
    from rag_baselines.c_tree import validate_tree
    validate_tree(tree, store)
    covered = {pid for node in tree['nodes'].values() for pid in node['passage_ids']}
    assert covered == set(store.indices)
    display(pd.DataFrame([{'id':node['id'],'path':' / '.join(node['path']),
                          'children':len(node['children']),'original_passages':len(node['passage_ids'])}
                         for node in tree['nodes'].values()]).head(30))
    print('All original passages reachable:', len(covered))
    print('Descriptor policy:', tree['descriptor_policy'])


### 다음 단계
`05_measure_abcd.ipynb`에서 동일한 질문·정답 근거·생성기로 A/B/C/D를 비교한다. C와 D는 이 동일한 Jev 준비 트리를 사용한다.
초기 분류체계와 임계값은 연구 설정이며 최적값으로 검증되지 않았다. 변경·튜닝은 개발셋에 한정하고 평가 전에 고정한다.
분류체계 작성 시간은 아직 측정하지 않았으며 준비 API 비용과 별개다.
[TypeSafe API](https://docs.typesafe.ai/api) · [모델·가격·문맥 제한](https://docs.typesafe.ai/models)